# 04 · Henry Hub price forecasting

**Target:** log change in the weekly average Henry Hub spot price, 1 and 4 weeks ahead.

**Benchmark:** a random walk (next price = today's price). Commodity prices are close to a random walk, so "beats the random walk with statistical significance" is the bar, not "has a low error".

**Leakage controls**
* Storage for week *t* is published the following Thursday, so features use storage lagged one week.
* Monthly LNG exports are published with roughly a two-month delay and are only used once available.
* Weather and storage "normals" use prior years only.
* Walk-forward: models are refit every 13 weeks on data whose targets were already realized at that date.

**Features:** price momentum (1/4/13-week returns, volatility), storage surplus vs. 5-yr average, injection vs. normal, national degree-day anomaly, LNG export level and growth, seasonality.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data, features, plotting
plotting.use_style()
pd.set_option("display.float_format", "{:,.2f}".format)
from src import price_models as pm
raw = data.load_raw()
dfp = features.build_price_frame(raw["henry_hub"], raw["storage_l48"], raw["lng_exports"], raw["weather_us"])
dfp[features.PRICE_FEATURES].describe().T

## Walk-forward evaluation

In [ ]:
results = {}
for h in (1, 4):
    wf = pm.walk_forward(dfp, h=h)
    results[h] = wf
    print(f"\n=== {h}-week horizon ===")
    display(pm.score_price(wf, h=h))

**Reading the table:** `RMSE vs RW %` below zero means the model beat the random walk. The Diebold–Mariano test (with Newey–West variance for overlapping horizons) says whether that difference is distinguishable from luck: negative statistic with p < 0.05 is a real edge.

In [ ]:
wf = results[4]
fig, ax = plt.subplots()
rw = wf[wf.model == "Random walk"].set_index("date")
ax.plot(rw.index, rw["price_actual"], color=plotting.NEUTRAL, lw=1.2, label="Actual (4 weeks later)")
g = wf[wf.model == "LightGBM (all features)"].set_index("date")
ax.plot(g.index, g["price_fcst"], color=plotting.MODEL_COLORS["LightGBM (all features)"], lw=1, label="LightGBM 4-week forecast")
ax.set_ylabel("$/MMBtu")
ax.set_title("Henry Hub: 4-week-ahead forecasts, out of sample")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(dfp.index, dfp["price"], color=plotting.SERIES[0], lw=1.2)
for d, label in config.EVENTS:
    d = pd.Timestamp(d)
    ax.axvline(d, color=plotting.INK_2, lw=.8, ls=":")
    ax.text(d, ax.get_ylim()[1] * .98, label, rotation=90, va="top", ha="right", fontsize=8, color=plotting.INK_2)
ax.set_ylabel("$/MMBtu")
ax.set_title("Henry Hub with major market events")
plt.show()

## What drives the model? (SHAP)
LightGBM's native TreeSHAP decomposes each prediction into feature contributions.

In [ ]:
cols = features.PRICE_FEATURES
lg = pm.LGBMPrice(cols)
tr = dfp.dropna(subset=cols + ["y_h4"])
lg.fit(tr, tr["y_h4"])
sh = lg.shap(tr)
imp = sh.abs().mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(imp.index, imp.values * 100, color=plotting.SERIES[0], height=.6)
ax.set_xlabel("Mean |SHAP| (percentage points of 4-week price change)")
ax.set_title("Feature importance, 4-week model")
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
plt.show()

In [ ]:
x = tr["storage_surplus_pct"]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, sh["storage_surplus_pct"] * 100, s=8, color=plotting.SERIES[0], alpha=.4)
ax.axhline(0, color=plotting.INK_2, lw=.8)
ax.set_xlabel("Storage vs. 5-yr avg (%)")
ax.set_ylabel("Contribution to forecast (pts)")
ax.set_title("How storage surplus moves the 4-week forecast")
plt.show()

## Current forecast

In [ ]:
rows = []
for h in (1, 4):
    for m in pm.model_zoo():
        f = pm.latest_forecast(dfp, m, h); f["model"] = m.name; rows.append(f)
pd.DataFrame(rows).pivot(index="model", columns="h", values="price_fcst").rename(
    columns={1: "1 week ahead $", 4: "4 weeks ahead $"})

## Conclusions
*Fill in after running on real data.* An honest result, even "fundamentals add little beyond a random walk at 1 week, with a modest edge at 4 weeks," is more credible to a hiring manager than an implausibly accurate price model.